# Hybrid MPC — regressão de $\cos(x)$

Neste notebook realizamos **apenas regressão** e treinamos **a mesma arquitetura** com a QNN **congelada** e **treinável**.

$$\hat f(x)=\sum_{j=1}^{M}\left[y_j(x)+1\right]\langle P_j\rangle^j$$

A QNN tem parâmetros independentes de $x$. A MLP recebe $x$ e produz os coeficientes $y_j(x)$.

**Como executar:** abra o notebook na pasta do projeto, instale com `pip install -e ".[notebook]"` (feito uma vez) e execute todas as células. Também funciona sem instalação ao executar a partir de `notebooks/`.

In [ ]:
# 1. Importações e localização do código da biblioteca.
from pathlib import Path
import sys
import numpy as np
import torch
import matplotlib.pyplot as plt

root = Path.cwd().resolve()
if not (root / "src").exists():
    root = root.parent
sys.path.insert(0, str(root / "src"))

from hybrid_mpc import Hybrid
from hybrid_mpc.loss.supervised import mse
from hybrid_mpc.training.train import train
from hybrid_mpc.training.save import save_run
from hybrid_mpc.analysis.metrics import rmse, mae
from hybrid_mpc.problems.regression_example import cosine_data

plt.rcParams.update({"font.size": 12, "axes.linewidth": 1.3})

In [ ]:
# 2. Parâmetros do experimento — altere apenas estas variáveis.
SEED = 1924
EPOCHS = 400
LR = 0.01
POINTS = 80
WIDTH = 32
HIDDEN_LAYERS = 2
N_QUBITS = 3
Q_LAYERS = 2
OBSERVABLES = ["ZII", "ZZI", "ZZZ"]  # ordem = expoentes 1, 2, 3
MPC_FUNCTION = "sequential"

print(f"Seed={SEED}, epochs={EPOCHS}, lr={LR}, points={POINTS}")
print(f"MLP: {HIDDEN_LAYERS}x{WIDTH}; QNN: {N_QUBITS} qubits, {Q_LAYERS} camadas")
print("Correlatores (em ordem):", OBSERVABLES)
print("MPC:", MPC_FUNCTION)

In [ ]:
# 3. Dados de treino e uma malha mais densa para avaliar a aproximação.
x_train, y_train = cosine_data(POINTS)
x_plot = torch.linspace(-torch.pi, torch.pi, 300).reshape(-1, 1)
y_exact = torch.cos(x_plot)

print("Treino:", x_train.shape, "| Avaliação:", x_plot.shape)

In [ ]:
# 4. Treina os dois modos. A seed igual garante a mesma inicialização.
results = {}

for name, trainable in [("Frozen", False), ("Trainable", True)]:
    torch.manual_seed(SEED)

    model = Hybrid(
        inputs=1,
        width=WIDTH, hidden_layers=HIDDEN_LAYERS,
        n_qubits=N_QUBITS, observables=OBSERVABLES,
        q_layers=Q_LAYERS, trainable=trainable,
        mpc_function=MPC_FUNCTION,
    )

    with torch.no_grad():
        q_initial = model.quantum().cpu().numpy().copy()

    history = train(model, lambda net: mse(net(x_train), y_train),
                    epochs=EPOCHS, lr=LR)

    with torch.no_grad():
        parts = model.components(x_plot)
        prediction = parts["output"].cpu().numpy().flatten()
        q_final = parts["quantum"].cpu().numpy().copy()
        terms = parts["terms"].cpu().numpy().copy()
        error_rmse = rmse(parts["output"], y_exact)
        error_mae = mae(parts["output"], y_exact)

    results[name] = {
        "model": model,
        "loss": history,
        "prediction": prediction,
        "q_initial": q_initial,
        "q_final": q_final,
        "terms": terms,
        "rmse": error_rmse,
        "mae": error_mae,
    }
    print(f"{name:9s}: RMSE={error_rmse:.6f} | MAE={error_mae:.6f} | final MSE={history[-1]:.6e}")

In [ ]:
# 5. Gráficos: aproximação, convergência e termos individuais do MPC.
fig, axes = plt.subplots(1, 3, figsize=(17, 4.7))
xx = x_plot.flatten().numpy()

axes[0].plot(xx, y_exact.flatten().numpy(), "k--", lw=2.5, label="cos(x)")
for name, result in results.items():
    axes[0].plot(xx, result["prediction"], lw=2, label=name)
axes[0].scatter(x_train.flatten(), y_train.flatten(), s=12, color="gray", alpha=0.4)
axes[0].set(title="Regression", xlabel="x", ylabel="f(x)")
axes[0].legend()

for name, result in results.items():
    axes[1].semilogy(result["loss"], lw=2, label=name)
axes[1].set(title="Training loss", xlabel="Epoch", ylabel="MSE")
axes[1].legend()

# Termos de apenas uma execução (modo treinável).
for j, name in enumerate(OBSERVABLES):
    axes[2].plot(xx, results["Trainable"]["terms"][:, j], lw=2,
                 label=f"{name} — power {j+1}")
axes[2].set(title="MPC contributions (trainable)", xlabel="x", ylabel="Term value")
axes[2].legend(fontsize=9)

for ax in axes:
    ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

In [ ]:
# 6. Valores esperados antes/depois do treino e as potências efetivas.
for name, result in results.items():
    print(f"\n{name}:")
    print(f"{'Pauli':<8} {'Power':>5} {'Initial <P>':>14} {'Final <P>':>14} {'Final <P>^j':>15}")
    for j, obs in enumerate(OBSERVABLES):
        qi = result["q_initial"][j]
        qf = result["q_final"][j]
        print(f"{obs:<8} {j+1:>5} {qi:>14.6f} {qf:>14.6f} {qf**(j+1):>15.6f}")

In [ ]:
# 7. Gráfico específico dos valores esperados e dos pesos polinomiais.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
positions = np.arange(len(OBSERVABLES))

for name, result in results.items():
    q = result["q_final"]
    powers = positions + 1
    axes[0].plot(positions, q, "o-", lw=2, label=name)
    axes[1].plot(positions, q**powers, "o-", lw=2, label=name)

axes[0].set(title="Pauli expectation values", ylabel="<P>")
axes[1].set(title="Sequential polynomial weights", ylabel="<P>^j")
for ax in axes:
    ax.set_xticks(positions, OBSERVABLES)
    ax.set(xlabel="Observable")
    ax.axhline(0, color="gray", lw=1)
    ax.grid(alpha=0.25)
    ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# 8. Salva os dois modelos e seus históricos, sem sobrescrever resultados antigos.
for name, result in results.items():
    folder = root / "runs" / "cosine_notebook" / f"{name.lower()}_seed_{SEED}"
    if folder.exists():
        print("SKIP (já existe):", folder)
        continue

    config = {
        "model": {
            "inputs": 1, "width": WIDTH,
            "hidden_layers": HIDDEN_LAYERS, "n_qubits": N_QUBITS,
            "observables": OBSERVABLES, "q_layers": Q_LAYERS,
            "trainable": name == "Trainable", "mpc_function": MPC_FUNCTION,
        },
        "train": {"seed": SEED, "epochs": EPOCHS, "lr": LR, "points": POINTS},
    }
    summary = {"rmse": result["rmse"], "mae": result["mae"],
               "final_loss": result["loss"][-1]}
    save_run(folder, config, result["model"], result["loss"], summary)
    print("Salvo:", folder)

## Interpretação importante

- A QNN é **independente da entrada**: $\langle P_j\rangle$ é um coeficiente global aprendido (ou congelado), não uma função de $x$.
- Para correlatores com valor próximo de zero, potências maiores podem diminuir muito a contribuição e também os gradientes.
- Potências **pares** tornam os valores esperados elevados à potência não negativos, mesmo que $\langle P_j\rangle$ seja negativo.
- O benchmark aqui é **didático**: como a MLP aprende os coeficientes e a QNN não recebe $x$, ele **não demonstra vantagem quântica**.
- Os pontos de avaliação formam uma malha mais densa no mesmo intervalo. Para estudar generalização, separe conjuntos de treino/teste ou use um intervalo distinto.